In [7]:
# 1. INSTALL DEPENDENCIES (Run this in a separate cell if using a Jupyter Notebook)
%pip install langchain_huggingface langchain_qdrant qdrant_client openai dotenv sentence-transformers -q


In [8]:
%pip install python-dotenv

In [6]:
# Prompt the user for their name
user_name = input("Please enter your name: ")

# Use the entered value
print(f"Hello, {user_name}! Welcome to Google Colab.")

Hello, ! Welcome to Google Colab.


In [10]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings  # Updated import
from langchain_qdrant import QdrantVectorStore

load_dotenv()  # Load environment variables from .env file
from openai import OpenAI
from qdrant_client import QdrantClient

# Client for Google Gemini API initialization from the OpenAI class
client = OpenAI(
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

# NEW FREE BGE EMBEDDINGS SYSTEM (Runs locally, automatically handles downloading)
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},  # Change to 'cuda' if you have an Nvidia GPU
    encode_kwargs={
        "normalize_embeddings": True
    },  # Essential for BGE cosine distance calculation
)

qclient = QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    check_compatibility=False,
)  # Qdrant local server URL

# Connect to the new BGE-compatible collection
# Note: You must run your ingestion/PDF-parsing script first using this new collection name
vector_db = QdrantVectorStore(
    embedding=embedding_model,
    client=qclient,  # Use the Qdrant client for connection
    # url="http://localhost:6333",  # Qdrant local server URL
    collection_name="chess_tactics_hf",  # Re-ingested collection name for 384 dimensions
)


    # Relevant chunks from the vector db
search_results = vector_db.similarity_search(
    "what do u know about this book", k=3
)  # k is the number of relevant chunks to retrieve

context = "\n\n\n".join(
    [
        f"Page Content:{result.page_content}\nPage Number: {result.metadata.get('page_label', 'N/A')}\nFile Location: {result.metadata.get('source', 'N/A')}"
        for result in search_results
    ]
)

SYSTEM_PROMPT = f"""You are a helpful assistant who answers questions based on the available context
retrieved from a PDF file along with page contents and page numbers.

You only answer the user based on the following context and navigate the
user to the open the right page number to know more.

Context:
{context}
"""

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "hi"},
    ],
)

print(f"🤖: {response.choices[0].message.content}\n")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🤖: Hello! I am here to help you with any questions regarding chess tactics and combinations based on the provided course document. 

You can start by exploring the resource, which begins on **page 1** of `tactics_course.pdf`, or by looking at specific game examples found on **pages 72 and 77**. 

How can I help you today?

